# LeetCode 3178 — Find the Child Who Has the Ball After K Seconds bằng Python

Link đề: https://leetcode.com/problems/find-the-child-who-has-the-ball-after-k-seconds/ · Độ khó: **Easy** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def numberOfChild(self, n: int, k: int) -> int:
        ...
```

Ghi chú: đề và ví dụ được chép/dịch theo đề gốc (bài giống hệt **2582. Pass the Pillow**); mọi đáp án trong notebook đã được kiểm chứng bằng code chạy thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` đứa trẻ xếp hàng đánh số `0..n-1`; bóng bắt đầu ở trẻ `0`, truyền sang phải mỗi giây một người, **chạm đầu mút thì đổi chiều**; sau `k` giây bóng ở trẻ nào?
2. Nắm các công cụ Python: vòng `for _ in range(k)`, biến **hướng** `direction = +1 / -1`, toán tử `%` và `//`, `divmod`, `list(range(a, b, -1))`, và `abs`.
3. Đi từ cách **mô phỏng từng giây** → **tìm chu kỳ** `2(n-1)` rồi dùng `%` → **gập chiều theo số lượt đi hết hàng** → **bảng đường đi một chu kỳ** → **công thức "sóng tam giác" bằng `abs`**.
4. Hiểu mấu chốt: chuyển động qua lại là **tuần hoàn**; bóng quay về trạng thái ban đầu sau `2(n-1)` giây, nên chỉ cần `k % (2(n-1))`.
5. Tránh các lỗi kinh điển: dùng chu kỳ sai (`n` hay `2n` thay vì `2n-2`), lệch một ở điều kiện `r < n` / `r <= n-1`, đổi chiều **sau** khi đã đi quá mút, và quên `n = 2` (chu kỳ chỉ là `2`).

## 2. Đề bài & ví dụ

Cho hai số nguyên dương `n` và `k`. Có `n` đứa trẻ đánh số từ `0` đến `n - 1` đứng thành hàng từ trái sang phải.

Ban đầu trẻ `0` cầm bóng và hướng truyền là sang **phải**. Sau mỗi giây, đứa trẻ đang cầm bóng truyền cho đứa kế bên. Khi bóng tới **một trong hai đầu** hàng (trẻ `0` hoặc trẻ `n - 1`), **hướng truyền bị đảo ngược**.

Trả về số hiệu của đứa trẻ nhận bóng sau `k` giây.

**Ví dụ 1**

- Input: `n = 3, k = 5`
- Output: `1`
- Giải thích: bóng đi `0 → 1 → 2 → 1 → 0 → 1`; sau 5 giây ở trẻ `1`.

**Ví dụ 2**

- Input: `n = 5, k = 6`
- Output: `2`
- Giải thích: bóng đi `0,1,2,3,4,3,2`; sau 6 giây ở trẻ `2`.

**Ví dụ 3**

- Input: `n = 4, k = 2`
- Output: `2`
- Giải thích: bóng đi `0,1,2`; sau 2 giây ở trẻ `2` (chưa tới đầu mút).

**Ràng buộc:** `2 <= n <= 50`, `1 <= k <= 50`.

In [1]:
examples = [
    (3, 5, 1),
    (5, 6, 2),
    (4, 2, 2),
]
for n, k, expected in examples:
    print(f"n={n}, k={k} -> kỳ vọng {expected}")

n=3, k=5 -> kỳ vọng 1
n=5, k=6 -> kỳ vọng 2
n=4, k=2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Vòng lặp `for _ in range(k)`

`range(k)` sinh `0, 1, ..., k-1`: đúng `k` lần lặp. Khi ta không cần giá trị biến đếm, quy ước đặt tên `_`. Ở bài này "mỗi giây" tương ứng "mỗi lần lặp".

In [2]:
count = 0
for _ in range(4):
    count += 1
print(count)                             # 4  (đúng 4 lần lặp)
print(list(range(4)))                    # [0, 1, 2, 3]

4
[0, 1, 2, 3]


### 3.2. Biến hướng `direction = +1 / -1`

Thay vì viết hai nhánh "đi phải"/"đi trái", ta lưu **một số** `direction` bằng `+1` (sang phải) hoặc `-1` (sang trái). Mỗi giây `pos += direction`. **Đảo chiều** chỉ là `direction = -direction`.

In [3]:
direction = 1
print(direction)                         # 1
direction = -direction
print(direction)                         # -1
direction = -direction
print(direction)                         # 1
pos = 2
pos += direction                         # đi 1 bước theo hướng hiện tại
print(pos)                               # 3

1
-1
1
3


### 3.3. Toán tử `%` (chia lấy dư), `//` (chia lấy nguyên) và `divmod`

- `a % m` là **số dư** khi chia `a` cho `m`, luôn trong `0..m-1` (với `m > 0`). Dùng để "quay vòng".
- `a // m` là **thương nguyên**.
- `divmod(a, m)` trả cả hai: `(a // m, a % m)`.

Đồng hồ là ví dụ quen thuộc: `15 % 12 = 3` (3 giờ chiều).

In [4]:
print(15 % 12, 15 // 12)                 # 3 1
print(divmod(15, 12))                    # (1, 3)
print(7 % 4, 8 % 4, 9 % 4)               # 3 0 1   (quay vòng theo chu kỳ 4)
print([x % 4 for x in range(10)])        # [0, 1, 2, 3, 0, 1, 2, 3, 0, 1]

3 1
(1, 3)
3 0 1
[0, 1, 2, 3, 0, 1, 2, 3, 0, 1]


### 3.4. Tạo dãy đếm xuôi và đếm ngược: `range` với bước âm

`range(a, b)` đi từ `a` đến `b-1`. `range(a, b, -1)` đi **ngược**, từ `a` xuống `b+1` (không gồm `b`). Ta sẽ dùng để viết đường đi của bóng: đi từ `0` lên `n-1` rồi từ `n-2` xuống `1`.

In [5]:
n = 5
print(list(range(n)))                    # [0, 1, 2, 3, 4]       đi tới đầu mút phải
print(list(range(n - 2, 0, -1)))         # [3, 2, 1]             đi ngược, bỏ đầu mút 4 và 0
print(list(range(n)) + list(range(n - 2, 0, -1)))   # [0, 1, 2, 3, 4, 3, 2, 1]   một chu kỳ

[0, 1, 2, 3, 4]
[3, 2, 1]
[0, 1, 2, 3, 4, 3, 2, 1]


### 3.5. `abs` và "sóng tam giác"

`abs(x)` là giá trị tuyệt đối. Hàm `f(x) = abs(x - c)` vẽ ra hình chữ `V`. Hiệu `c - abs(x - c)` vẽ ra hình chữ `Λ` (lên rồi xuống): đúng dạng quỹ đạo của quả bóng!

In [6]:
c = 4
print([c - abs(x - c) for x in range(9)])      # [0, 1, 2, 3, 4, 3, 2, 1, 0]  lên rồi xuống

[0, 1, 2, 3, 4, 3, 2, 1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `n = 4` (trẻ `0,1,2,3`). Viết trẻ cầm bóng sau từng giây:

| giây | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| trẻ | 0 | 1 | 2 | 3 | 2 | 1 | 0 | 1 | 2 | 3 |

Nhìn bảng: sau giây `6` bóng **quay về** trẻ `0` với hướng phải — y như giây `0`. Bóng lặp lại sau `6 = 2 × (4 - 1)` giây. Tức là: để biết trạng thái sau `k` giây, chỉ cần biết `k` **dư bao nhiêu** khi chia cho `6`.

Khái quát: bóng đi từ `0` đến `n-1` mất `n-1` giây, rồi quay lại `0` mất `n-1` giây nữa. **Chu kỳ = `2(n-1)`**.

In [7]:
n = 4
period = 2 * (n - 1)
print("chu kỳ =", period)
path = []
pos, direction = 0, 1
for second in range(10):
    path.append(pos)
    pos += direction
    if pos == 0 or pos == n - 1:
        direction = -direction
print(path)                              # [0, 1, 2, 3, 2, 1, 0, 1, 2, 3]
print([path[t] == path[t % period] for t in range(10)])   # toàn True: path[t] = path[t % chu kỳ]

chu kỳ = 6
[0, 1, 2, 3, 2, 1, 0, 1, 2, 3]
[True, True, True, True, True, True, True, True, True, True]


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng giây

**Ý tưởng.** Làm đúng như đề: giữ vị trí `pos` và hướng `direction`. Lặp `k` lần: `pos += direction`; nếu `pos` chạm `0` hoặc `n - 1` thì đảo hướng.

Điều quan trọng là **đảo hướng ngay sau khi bóng tới đầu mút** (để bước kế tiếp đi theo hướng mới), chứ không phải khi đã đi quá mút.

In [8]:
def number_of_child_simulate(n: int, k: int) -> int:
    pos, direction = 0, 1
    for _ in range(k):
        pos += direction                 # truyền bóng cho trẻ kế bên
        if pos == 0 or pos == n - 1:     # tới đầu mút -> đổi chiều cho các lượt sau
            direction = -direction
    return pos


for n, k, expected in examples:
    print(number_of_child_simulate(n, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
2 (kỳ vọng 2 )


**Độ phức tạp (Cách 1).** Lặp đúng `k` lần, mỗi lần `O(1)` ⇒ **Time `O(k)`**, **Space `O(1)`**. Với `k <= 50` quá đủ.

**Nút thắt:** nếu `k` rất lớn (ví dụ `10^9`) thì mô phỏng chậm. Vì chuyển động **tuần hoàn**, ta có thể "nhảy" thẳng bằng phép chia dư.

## 6. Ý tưởng thuật toán / bước đột phá

Vì sau mỗi `2(n-1)` giây bóng trở lại đúng vị trí và hướng ban đầu, ta thay `k` bằng `r = k % (2(n-1))`, `r` nằm trong `0 .. 2(n-1)-1`. Rồi đọc vị trí ở giây `r` của **một chu kỳ**:

- Nếu `r <= n - 1`: bóng còn **đang đi sang phải**, vị trí là `r`.
- Nếu `r > n - 1`: bóng đã **quay đầu**, đã đi `r - (n-1)` bước lùi từ `n-1`, vị trí là `(n-1) - (r - (n-1)) = 2(n-1) - r`.

Chỉ một phép `%` và một phép so sánh: `O(1)`.

In [9]:
n, k = 5, 6
period = 2 * (n - 1)
r = k % period
print("chu kỳ =", period, ", r =", r)    # 8 , 6
if r <= n - 1:
    print("đang đi phải, vị trí =", r)
else:
    print("đang đi trái, vị trí =", period - r)   # 8 - 6 = 2

chu kỳ = 8 , r = 6
đang đi trái, vị trí = 2


## 7. Các cách tiếp theo

### Cách 2 — Chu kỳ `2(n-1)` và `%`

**Giải thích từng đoạn.**

1. `period = 2 * (n - 1)`: số giây để bóng đi hết hàng rồi về lại `0`.
2. `r = k % period`: giây tương đương trong chu kỳ đầu.
3. `r <= n - 1` → đáp án `r` (đi phải); ngược lại `period - r` (đi trái).

In [10]:
def number_of_child_period(n: int, k: int) -> int:
    period = 2 * (n - 1)
    r = k % period
    if r <= n - 1:
        return r                         # nửa chu kỳ đầu: đang đi sang phải
    return period - r                    # nửa chu kỳ sau: đang đi về trái


for n, k, expected in examples:
    print(number_of_child_period(n, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
2 (kỳ vọng 2 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionPeriod:
    def numberOfChild(self, n: int, k: int) -> int:
        period = 2 * (n - 1)
        r = k % period
        return r if r <= n - 1 else period - r


print(SolutionPeriod().numberOfChild(5, 6))        # 2

2


**Độ phức tạp Cách 2:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 3 — Đếm số lượt "đi hết hàng" bằng `divmod`

**Giải thích.** Mỗi lượt đi từ đầu này sang đầu kia mất `n - 1` giây. Tính `full, rem = divmod(k, n - 1)`: `full` là số lượt **đã đi trọn**, `rem` là số bước đi thêm trong lượt hiện tại.

- `full` **chẵn**: lượt hiện tại đi **sang phải**, từ `0`: vị trí `rem`.
- `full` **lẻ**: lượt hiện tại đi **sang trái**, từ `n-1`: vị trí `(n-1) - rem`.

Cách nghĩ này nhấn mạnh "mỗi lượt đảo chiều một lần".

In [12]:
def number_of_child_divmod(n: int, k: int) -> int:
    full, rem = divmod(k, n - 1)         # full lượt đi trọn, rem bước lẻ
    if full % 2 == 0:
        return rem                       # lượt chẵn: đi phải từ 0
    return (n - 1) - rem                 # lượt lẻ: đi trái từ n-1


for n, k, expected in examples:
    print(number_of_child_divmod(n, k), "(kỳ vọng", expected, ")")
print(divmod(6, 4))                      # n=5, k=6: (1, 2) -> lượt lẻ -> 4 - 2 = 2

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
(1, 2)


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Bảng đường đi một chu kỳ rồi tra bằng chỉ số

**Giải thích.** Dựng sẵn `path` = một chu kỳ: `0, 1, ..., n-1, n-2, ..., 1`. Độ dài `n + (n-2) = 2n - 2 = 2(n-1)`. Đáp án là `path[k % len(path)]`. Cách này **dễ chứng minh đúng bằng mắt** và rất hợp để học; với `n = 2` thì `path = [0, 1]` (đoạn `range(0, 0, -1)` rỗng) — vẫn đúng.

In [13]:
def number_of_child_table(n: int, k: int) -> int:
    path = list(range(n)) + list(range(n - 2, 0, -1))    # một chu kỳ của quỹ đạo
    return path[k % len(path)]


for n, k, expected in examples:
    print(number_of_child_table(n, k), "(kỳ vọng", expected, ")")
print(list(range(2)) + list(range(0, 0, -1)))           # n=2 -> [0, 1]

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
[0, 1]


**Độ phức tạp Cách 4:** dựng `path` tốn `O(n)` ⇒ **Time `O(n)`**, **Space `O(n)`**.

### Cách 5 — Công thức "sóng tam giác" bằng `abs`

**Giải thích.** Đặt `x = k % (2(n-1))`. Vị trí bóng là `(n-1) - abs((n-1) - x)`. Kiểm tra: `x <= n-1` ⇒ `abs((n-1)-x) = (n-1)-x` ⇒ vị trí `x`. `x > n-1` ⇒ `abs = x-(n-1)` ⇒ vị trí `2(n-1) - x`. Đúng hệt hai nhánh của Cách 2, gộp thành một dòng.

In [14]:
def number_of_child_abs(n: int, k: int) -> int:
    x = k % (2 * (n - 1))
    return (n - 1) - abs((n - 1) - x)


for n, k, expected in examples:
    print(number_of_child_abs(n, k), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
2 (kỳ vọng 2 )


**Độ phức tạp Cách 5:** **Time `O(1)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 1 (mô phỏng) với `n = 5, k = 6` và Cách 2 (chu kỳ) với `n = 4, k = 8`.

Bảng trace Cách 1 với `n = 5, k = 6` (đầu mút là `0` và `4`):

| giây | pos trước | direction | pos sau | tới đầu mút? | direction mới |
|---|---|---|---|---|---|
| 1 | 0 | +1 | 1 | không | +1 |
| 2 | 1 | +1 | 2 | không | +1 |
| 3 | 2 | +1 | 3 | không | +1 |
| 4 | 3 | +1 | 4 | **có (n-1)** | **-1** |
| 5 | 4 | -1 | 3 | không | -1 |
| 6 | 3 | -1 | 2 | không | -1 |

Đáp án `2`.

In [15]:
def trace_simulate(n: int, k: int) -> int:
    pos, direction = 0, 1
    print(f"n = {n}, k = {k}; bắt đầu pos = {pos}, direction = {direction:+d}")
    for second in range(1, k + 1):
        before = pos
        pos += direction
        note = ""
        if pos == 0 or pos == n - 1:
            direction = -direction
            note = "  <- tới đầu mút, đổi chiều"
        print(f"  giây {second}: {before} -> {pos}, direction mới = {direction:+d}{note}")
    return pos


assert trace_simulate(5, 6) == 2

n = 5, k = 6; bắt đầu pos = 0, direction = +1
  giây 1: 0 -> 1, direction mới = +1
  giây 2: 1 -> 2, direction mới = +1
  giây 3: 2 -> 3, direction mới = +1
  giây 4: 3 -> 4, direction mới = -1  <- tới đầu mút, đổi chiều
  giây 5: 4 -> 3, direction mới = -1
  giây 6: 3 -> 2, direction mới = -1


In [16]:
def trace_period(n: int, k: int) -> int:
    period = 2 * (n - 1)
    r = k % period
    print(f"n = {n}, k = {k}: chu kỳ = 2*({n}-1) = {period}, r = {k} % {period} = {r}")
    if r <= n - 1:
        ans = r
        print(f"  r <= n-1 = {n - 1}: đang đi phải, đáp án = r = {ans}")
    else:
        ans = period - r
        print(f"  r > n-1 = {n - 1}: đã quay đầu, đáp án = {period} - {r} = {ans}")
    return ans


assert trace_period(4, 8) == 2           # path = 0,1,2,3,2,1 ; r = 8 % 6 = 2 -> 2
assert trace_period(3, 5) == 1           # chu kỳ 4 ; r = 1 -> 1
assert trace_period(5, 6) == 2           # chu kỳ 8 ; r = 6 > 4 -> 8 - 6 = 2

n = 4, k = 8: chu kỳ = 2*(4-1) = 6, r = 8 % 6 = 2
  r <= n-1 = 3: đang đi phải, đáp án = r = 2
n = 3, k = 5: chu kỳ = 2*(3-1) = 4, r = 5 % 4 = 1
  r <= n-1 = 2: đang đi phải, đáp án = r = 1
n = 5, k = 6: chu kỳ = 2*(5-1) = 8, r = 6 % 8 = 6
  r > n-1 = 4: đã quay đầu, đáp án = 8 - 6 = 2


Với `n = 4, k = 8`: chu kỳ `6`; `8 % 6 = 2`; `2 <= 3` nên đáp án `2`. Kiểm tra bảng ở mục 4: giây `8` ở trẻ `2` ✓.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (chu kỳ và `%`): `O(1)`, ngắn, dễ giải thích.

In [17]:
class Solution:
    def numberOfChild(self, n: int, k: int) -> int:
        period = 2 * (n - 1)             # bóng đi hết hàng rồi về lại 0 mất 2(n-1) giây
        r = k % period                   # thời điểm tương đương trong chu kỳ đầu
        if r <= n - 1:
            return r                     # đang đi sang phải
        return period - r                # đang đi về trái


sol = Solution()
for n, k, expected in examples:
    print(f"n={n}, k={k} ->", sol.numberOfChild(n, k))

n=3, k=5 -> 1
n=5, k=6 -> 2
n=4, k=2 -> 2


## 10. Vì sao thuật toán đúng

- **Tuần hoàn:** sau `2(n-1)` giây bóng ở đúng trẻ `0` và đang hướng sang phải (giống hệt giây `0`), mọi chuyển động sau đó lặp lại ⇒ vị trí ở giây `k` bằng vị trí ở giây `k % 2(n-1)`.
- **Nửa đầu chu kỳ (`0 <= r <= n-1`):** bóng đi thẳng sang phải mỗi giây một người, bắt đầu ở `0`, nên ở giây `r` là trẻ `r`. Tại `r = n-1` bóng tới đầu mút phải (và sẵn sàng đảo chiều).
- **Nửa sau chu kỳ (`n-1 < r < 2(n-1)`):** bóng đã quay đầu và lùi `r-(n-1)` bước từ `n-1`, nên ở `(n-1)-(r-(n-1)) = 2(n-1) - r`. Khi `r` đạt `2(n-1)` (tức `0` sau `%`) bóng về trẻ `0` — được xử lý ở nhánh `r = 0`.
- **`n = 2`:** chu kỳ `2`; `r = 0` → `0`, `r = 1` → `1`. Hợp lý: bóng nảy qua lại giữa hai trẻ.
- **Hai nhánh gặp nhau liền mạch** tại `r = n-1`: cả `r` lẫn `period - r` đều bằng `n - 1`. Vì vậy viết `r <= n - 1` hay `r < n - 1` cho nhánh đầu đều cho kết quả đúng; điều quan trọng là công thức của nhánh sau phải đúng (xem mục 13).

## 11. Phân tích Time / Space Complexity

Gọi `n, k` như đề.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng giây | `O(k)` | `O(1)` |
| 2. Chu kỳ và `%` | `O(1)` | `O(1)` |
| 3. `divmod(k, n-1)` và chẵn/lẻ | `O(1)` | `O(1)` |
| 4. Bảng một chu kỳ | `O(n)` | `O(n)` |
| 5. Sóng tam giác `abs` | `O(1)` | `O(1)` |

Cách 1 phụ thuộc `k`; nếu `k` lên đến `10^9` thì chỉ các cách `O(1)` dùng được.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
import random

sol = Solution()
impls = {
    "simulate": number_of_child_simulate,
    "period": number_of_child_period,
    "divmod": number_of_child_divmod,
    "table": number_of_child_table,
    "abs": number_of_child_abs,
    "leetcode": sol.numberOfChild,
}


def check_all(n, k, expected, label):
    for key, fn in impls.items():
        actual = fn(n, k)
        assert actual == expected, (label, key, n, k, actual, expected)
    print("PASS", label)


for idx, (n, k, expected) in enumerate(examples, 1):
    check_all(n, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all(2, 1, 1, "n=2, k=1: sang trẻ 1")
check_all(2, 2, 0, "n=2, k=2: nảy về 0")
check_all(2, 3, 1, "n=2, k=3")
check_all(2, 50, 0, "n=2, k=50 (chẵn)")
check_all(3, 1, 1, "n=3, k=1")
check_all(3, 2, 2, "n=3, k=2: tới đầu mút phải")
check_all(3, 4, 0, "n=3, k=4: đúng một chu kỳ, về 0")
check_all(4, 3, 3, "n=4, k=3: đúng đầu mút phải")
check_all(4, 6, 0, "n=4, k=6: đúng một chu kỳ")
check_all(4, 9, 3, "n=4, k=9: sang chu kỳ thứ hai")
check_all(5, 4, 4, "n=5, k=4: tới đầu mút phải")
check_all(5, 8, 0, "n=5, k=8: đúng một chu kỳ")
check_all(50, 49, 49, "n=50, k=49: tới đầu mút phải")
check_all(50, 50, 48, "n=50, k=50: vừa quay đầu")
check_all(6, 50, 0, "n=6, k=50: chu kỳ 10, r = 0")

PASS n=2, k=1: sang trẻ 1
PASS n=2, k=2: nảy về 0
PASS n=2, k=3
PASS n=2, k=50 (chẵn)
PASS n=3, k=1
PASS n=3, k=2: tới đầu mút phải
PASS n=3, k=4: đúng một chu kỳ, về 0
PASS n=4, k=3: đúng đầu mút phải
PASS n=4, k=6: đúng một chu kỳ
PASS n=4, k=9: sang chu kỳ thứ hai
PASS n=5, k=4: tới đầu mút phải
PASS n=5, k=8: đúng một chu kỳ
PASS n=50, k=49: tới đầu mút phải
PASS n=50, k=50: vừa quay đầu
PASS n=6, k=50: chu kỳ 10, r = 0


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) dùng **chu kỳ `n`** (hoặc `2n`) thay vì `2(n-1)`: đầu mút bị tính **hai lần**; (b) chỉ dùng `k % n` rồi coi như bóng chuyền vòng tròn.

In [20]:
def wrong_period_2n(n, k):
    period = 2 * n                       # SAI: đầu mút bị tính hai lần
    r = k % period
    return r if r < n else period - 1 - r


def wrong_circle(n, k):
    return k % n                         # SAI: coi như truyền vòng tròn, không quay đầu


print(wrong_period_2n(3, 5), "!=", Solution().numberOfChild(3, 5))
print(wrong_circle(3, 5), "!=", Solution().numberOfChild(3, 5))
assert wrong_period_2n(3, 5) != Solution().numberOfChild(3, 5)
assert wrong_circle(3, 5) != Solution().numberOfChild(3, 5)

0 != 1
2 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Với mọi `n` trong `2..50` và `k` trong `1..50` (vét cạn **toàn bộ** miền ràng buộc, `49 × 50 = 2450` cặp) ta so tất cả cách với mô phỏng. Sau đó thử ngẫu nhiên `k` lớn (ngoài ràng buộc, chỉ để kiểm chứng toán) với các cách `O(1)`. Thêm thuộc tính: kết quả luôn trong `0..n-1`, và hai thời điểm liên tiếp cho kết quả hơn kém nhau đúng `1` (bóng chỉ truyền cho người kế bên).

In [21]:
cnt = 0
for n in range(2, 51):
    for k in range(1, 51):
        ref = number_of_child_simulate(n, k)
        for key, fn in impls.items():
            assert fn(n, k) == ref, (key, n, k)
        assert 0 <= ref <= n - 1
        cnt += 1
print("PASS vét cạn", cnt, "cặp (n, k) trong ràng buộc")

rng = random.Random(3178)
for trial in range(2000):
    n = rng.randint(2, 50)
    k = rng.randint(1, 1000)                       # k lớn hơn ràng buộc để kiểm chứng công thức
    ref = number_of_child_simulate(n, k)
    for key in ["period", "divmod", "table", "abs", "leetcode"]:
        assert impls[key](n, k) == ref, (key, n, k)
print("PASS 2000 cặp (n, k) ngẫu nhiên với k tới 1000")

for n in range(2, 51):
    for k in range(0, 200):
        assert abs(Solution().numberOfChild(n, k + 1) - Solution().numberOfChild(n, k)) == 1, (n, k)
print("PASS tính chất: mỗi giây bóng dịch đúng 1 vị trí")

PASS vét cạn 2450 cặp (n, k) trong ràng buộc
PASS 2000 cặp (n, k) ngẫu nhiên với k tới 1000
PASS tính chất: mỗi giây bóng dịch đúng 1 vị trí


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng chu kỳ `n` hoặc `2n` thay vì `2(n-1)`

In [22]:
def sai_chu_ky_n(n, k):
    r = k % n                            # SAI: chu kỳ không phải n
    return r


for n, k in [(3, 5), (4, 6), (5, 8)]:
    print(f"n={n}, k={k}: sai = {sai_chu_ky_n(n, k)}, đúng = {Solution().numberOfChild(n, k)}")

n=3, k=5: sai = 2, đúng = 1
n=4, k=6: sai = 2, đúng = 0
n=5, k=8: sai = 3, đúng = 0


**Vì sao sai:** bóng phải đi hết hàng (`n-1` giây) **rồi quay về** (`n-1` giây). Đầu mút `0` và `n-1` mỗi cái chỉ được "chạm" một lần mỗi chu kỳ nên chu kỳ là `2n - 2`, không phải `n` hay `2n`. **Cách sửa:** `period = 2 * (n - 1)`.

### Lỗi 2: Đổi chiều **sau khi** đã đi quá đầu mút (kiểm tra lệch bước)

In [23]:
def sai_doi_chieu_tre(n, k):
    pos, direction = 0, 1
    for _ in range(k):
        if pos == 0 or pos == n - 1:     # SAI: kiểm tra TRƯỚC khi đi, ở bước đầu pos=0 đã đảo chiều
            direction = -direction
        pos += direction
    return pos


print("sai:", sai_doi_chieu_tre(3, 5), "| đúng:", Solution().numberOfChild(3, 5))

sai: -5 | đúng: 1


**Vì sao sai:** ở bước đầu tiên `pos = 0` đã là đầu mút nên hướng bị đảo thành `-1` ngay và bóng đi ra ngoài hàng (`-1`). Phải đảo hướng **sau khi** bóng vừa tới đầu mút (như Cách 1). **Cách sửa:** `pos += direction` trước, rồi `if pos == 0 or pos == n - 1: direction = -direction`.

### Lỗi 3: Lệch một ở nhánh `r < n` hay `r <= n - 1`; nhầm công thức nửa sau

In [24]:
def sai_cong_thuc_nua_sau(n, k):
    period = 2 * (n - 1)
    r = k % period
    if r <= n - 1:
        return r
    return period - r - 1                # SAI: trừ thêm 1 do nhầm với chu kỳ 2n


def sai_nhanh_ngat(n, k):
    period = 2 * (n - 1)
    r = k % period
    if r < n - 1:                        # dùng < cũng ĐÚNG vì tại r = n-1 hai nhánh trùng nhau ...
        return r
    return n - 1 - (r - n)               # ... nhưng công thức này SAI: phải là (n-1) - (r-(n-1))


for n, k in [(5, 6), (4, 4)]:
    print(f"n={n}, k={k}: sai1 = {sai_cong_thuc_nua_sau(n, k)}, sai2 = {sai_nhanh_ngat(n, k)}, đúng = {Solution().numberOfChild(n, k)}")

n=5, k=6: sai1 = 1, sai2 = 3, đúng = 2
n=4, k=4: sai1 = 1, sai2 = 3, đúng = 2


**Vì sao sai:** nửa sau chu kỳ bóng lùi `r - (n-1)` bước từ `n-1` ⇒ vị trí `(n-1) - (r-(n-1)) = 2(n-1) - r = period - r`. Trừ thêm `1` hoặc viết `r - n` làm lệch một. Còn dùng `r < n-1` hay `r <= n-1` cho nhánh đầu đều đúng vì tại `r = n-1` hai công thức `r` và `period - r` cùng cho `n-1`. **Cách sửa:** luôn **thử `r = n - 1`** (đầu mút) để kiểm tra công thức.

### Lỗi 4: Quên `n = 2`

In [25]:
def sai_n_bang_2(n, k):
    path = list(range(n)) + list(range(n - 1, 0, -1))   # SAI: range(n-1, 0, -1) bắt đầu lại từ n-1 (lặp đầu mút)
    return path[k % len(path)]


print("sai:", [sai_n_bang_2(2, k) for k in range(1, 6)], "| đúng:", [Solution().numberOfChild(2, k) for k in range(1, 6)])

sai: [1, 1, 0, 1, 1] | đúng: [1, 0, 1, 0, 1]


**Vì sao sai:** nếu dựng đường đi bằng `range(n-1, 0, -1)` thì đầu mút `n-1` xuất hiện hai lần liên tiếp (`..., n-1, n-1, ...`) tức bóng "đứng yên" một giây. Phải bắt đầu đường lùi từ `n-2`: `range(n-2, 0, -1)`. Với `n = 2` đoạn này rỗng và chu kỳ chỉ là `[0, 1]`. **Cách sửa:** `list(range(n)) + list(range(n - 2, 0, -1))`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đi qua lại / nảy giữa hai đầu / quay đầu khi chạm biên" ⇒ **chuyển động tuần hoàn**: tìm chu kỳ rồi dùng `%`.
- Chu kỳ nảy qua lại trên `n` ô là `2(n-1)` (không phải `2n`) vì **đầu mút chỉ được tính một lần**.
- Khi nghi ngờ công thức, **viết một mô phỏng ngắn** rồi so tất cả `(n, k)` nhỏ bằng `assert` (vét cạn như mục 12.3).
- Công thức sóng tam giác `(n-1) - abs((n-1) - x)` cô đọng nhưng khó đọc; nộp bài nên chọn hai nhánh `if` rõ ràng.
- Kiểm tra luôn các điểm **biên**: `k = 0` (nếu cho phép), `r = n-1`, `r = 0`, `n = 2`.
- Bài này **giống hệt LeetCode 2582 Pass the Pillow**: nhớ lời giải chung cho cả hai.

## 15. Biến thể của bài

1. **k rất lớn** (`10^9`): chỉ cách `O(1)` dùng được.
2. **Trả về toàn bộ đường đi** của bóng trong `k` giây.
3. **Bắt đầu ở trẻ `s`** với hướng cho trước (không phải trẻ `0`).
4. **Đếm số lần** trẻ `i` giữ bóng trong `k` giây đầu.
5. **Pass the Pillow (2582)**, **Find the Winner of the Circular Game (1823)** (truyền vòng tròn, loại dần), **Robot Bounded In Circle (1041)** (chuyển động tuần hoàn).

In [26]:
def ball_path(n, k):
    # đường đi của bóng từ giây 0 đến giây k
    pos, d, out = 0, 1, [0]
    for _ in range(k):
        pos += d
        if pos == 0 or pos == n - 1:
            d = -d
        out.append(pos)
    return out


def number_of_child_from(n, k, start, direction):
    # bóng bắt đầu ở trẻ start, hướng direction (+1 phải, -1 trái)
    pos, d = start, direction
    for _ in range(k):
        if pos + d < 0 or pos + d > n - 1:   # sắp ra ngoài -> đổi chiều trước khi đi
            d = -d
        pos += d
        if pos == 0 or pos == n - 1:
            d = -d
    return pos


def times_holding(n, k, child):
    return ball_path(n, k).count(child)


print(ball_path(5, 6))                               # [0, 1, 2, 3, 4, 3, 2]
print(number_of_child_from(5, 3, 3, 1))              # 3 -> 4 (đổi chiều) -> 3 -> 2 => 2
print(times_holding(3, 8, 1))                        # path 0,1,2,1,0,1,2,1,0 -> trẻ 1: 4 lần
assert ball_path(5, 6) == [0, 1, 2, 3, 4, 3, 2]
assert number_of_child_from(5, 3, 3, 1) == 2
assert number_of_child_from(5, 6, 0, 1) == Solution().numberOfChild(5, 6)
assert times_holding(3, 8, 1) == 4

[0, 1, 2, 3, 4, 3, 2]
2
4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đi từng giây, đổi chiều ở đầu mút | `O(k)` | `O(1)` | `k` nhỏ, dễ hiểu, dùng làm đối chiếu |
| 2. Chu kỳ `%` | `r = k % 2(n-1)`, hai nhánh | `O(1)` | `O(1)` | **nên nộp** |
| 3. `divmod` | đếm lượt đi hết hàng, chẵn/lẻ | `O(1)` | `O(1)` | khi nghĩ theo "lượt" |
| 4. Bảng chu kỳ | dựng `path`, tra `path[k % len]` | `O(n)` | `O(n)` | học, dễ chứng minh |
| 5. `abs` | sóng tam giác một dòng | `O(1)` | `O(1)` | code ngắn, cần giải thích kỹ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `17 % 6`, `divmod(17, 6)`, `list(range(5, 0, -2))`, `Solution().numberOfChild(2, 7)`, `Solution().numberOfChild(4, 4)`.

**Bài 2 (tính tay).** Với `n = 6, k = 13`: tính chu kỳ, `r`, rồi đáp án. Sau đó viết ra đường đi `0..13` để kiểm tra.

**Bài 3 (điền TODO).** Hoàn thành `number_of_child_todo(n, k)` theo Cách 2.

In [27]:
def number_of_child_todo(n, k):
    # TODO 1: period = 2 * (n - 1)
    # TODO 2: r = k % period
    # TODO 3: nếu r <= n - 1 trả về r; ngược lại trả về period - r
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `number_of_child_divmod2(n, k)` theo Cách 3 (dùng `divmod(k, n - 1)` và kiểm tra số lượt chẵn/lẻ), **không** dùng `period`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `first_time_at_end(n)` trả về số giây tối thiểu để bóng lần đầu tới trẻ `n - 1`. (b) Viết `number_of_child_start(n, k, start)`: bóng bắt đầu ở trẻ `start` đang hướng **phải** (nếu `start = n - 1` thì lượt đầu đi trái); trả về trẻ cầm bóng sau `k` giây. Gợi ý: đưa về bài gốc bằng cách tính "giây ảo" `start` của chu kỳ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `5`, `(2, 5)`, `[5, 3, 1]`, `1` (chu kỳ 2, `7 % 2 = 1`), `2` (chu kỳ 6, `4 % 6 = 4 > 3` ⇒ `6 - 4 = 2`). Bài 3: ba dòng, hai nhánh `if`.
- **Mức 2 (vừa):** Bài 2: chu kỳ `10`, `r = 3`, `3 <= 5` nên đáp án `3`; đường đi `0,1,2,3,4,5,4,3,2,1,0,1,2,3` ✓. Bài 4: `full, rem = divmod(k, n - 1)`; `full` chẵn trả `rem`, lẻ trả `n - 1 - rem`.
- **Mức 3 (sâu):** Bài 5(a): đáp án `n - 1`. Bài 5(b): coi như bóng đang ở "giây ảo" `start` của chu kỳ chuẩn (đi phải từ `0`; riêng `start = n - 1` thì giây ảo `n - 1` và lượt kế tiếp tự đi trái). Khi đó `t = (start + k) % period` rồi tra như Cách 2.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 17 % 6 == 5
assert divmod(17, 6) == (2, 5)
assert list(range(5, 0, -2)) == [5, 3, 1]
assert Solution().numberOfChild(2, 7) == 1
assert Solution().numberOfChild(4, 4) == 2

# Bài 2
period = 2 * (6 - 1)
r = 13 % period
assert (period, r) == (10, 3)
assert Solution().numberOfChild(6, 13) == 3
assert ball_path(6, 13) == [0, 1, 2, 3, 4, 5, 4, 3, 2, 1, 0, 1, 2, 3]


# Bài 3
def number_of_child_todo_ans(n, k):
    period = 2 * (n - 1)
    r = k % period
    if r <= n - 1:
        return r
    return period - r


for n, k, expected in examples:
    assert number_of_child_todo_ans(n, k) == expected


# Bài 4
def number_of_child_divmod2(n, k):
    full, rem = divmod(k, n - 1)
    return rem if full % 2 == 0 else (n - 1) - rem


for n in range(2, 51):
    for k in range(1, 51):
        assert number_of_child_divmod2(n, k) == Solution().numberOfChild(n, k)


# Bài 5(a)
def first_time_at_end(n):
    return n - 1


for n in range(2, 51):
    path = ball_path(n, 3 * n)
    assert path.index(n - 1) == first_time_at_end(n)


# Bài 5(b)
def number_of_child_start(n, k, start):
    period = 2 * (n - 1)
    t = (start + k) % period            # giây ảo trong chu kỳ chuẩn (hướng phải tại start, hoặc tại mút phải thì lượt tiếp theo đi trái)
    return t if t <= n - 1 else period - t


for n in range(2, 12):
    for start in range(n):
        for k in range(0, 40):
            assert number_of_child_start(n, k, start) == number_of_child_from(n, k, start, 1), (n, k, start)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chuyển động nảy qua lại: chu kỳ `2(n-1)`)

```python
class Solution:
    def numberOfChild(self, n: int, k: int) -> int:
        period = 2 * (n - 1)
        r = k % period
        return r if r <= n - 1 else period - r
```

**Checklist**

- [ ] Chu kỳ là `2 * (n - 1)` (đầu mút chỉ tính một lần), không phải `n` hay `2n`?
- [ ] Nhánh đi phải trả `r`, nhánh đi trái trả `period - r`; đã thử `r = n - 1` và `r = 0`?
- [ ] Nếu mô phỏng: đảo hướng **sau** khi tới đầu mút, không phải trước?
- [ ] Đã thử `n = 2` (chu kỳ `2`) và `k` bằng đúng một chu kỳ?
- [ ] Đã thử ví dụ `(3, 5) → 1`, `(5, 6) → 2`, `(4, 2) → 2`?
- [ ] Skeleton đúng `class Solution` / `def numberOfChild(self, n: int, k: int) -> int`?